In [1]:
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, Polygon
from pyrsimg import coor2coor  

In [2]:
input_file = 'data\hma-extent\HMA\hma_gtng_202307_subregions.gpkg' 
tb_area = gpd.read_file(input_file)

# Ensure WGS84 projection
if tb_area.crs != "EPSG:4326":
    tb_area = tb_area.to_crs("EPSG:4326")

minx, miny, maxx, maxy = tb_area.total_bounds
print(f"Study Area Bounds: lon[{minx:.2f}, {maxx:.2f}], lat[{miny:.2f}, {maxy:.2f}]")

# Parameters setting
buffer_m = 10000       # UTM buffer size: 10 km
buffer_deg = 0.1       # WGS84 buffer size: 0.1 degrees
dx_utm = 100000        # UTM grid width: 100 km
dy_utm = 100000        # UTM grid height: 100 km

Study Area Bounds: lon[67.00, 104.80], lat[26.76, 46.00]


In [3]:
def get_utm_epsg(lon, lat):
    """
    des:
        Calculate the corresponding UTM projection EPSG number based on latitude and longitude.
    para:
        lon : float 
        lat : float
    return:
        str: Formatted EPSG number
    """
    zone_number = math.floor((lon + 180) / 6) + 1
    epsg_code = 32600 + zone_number if lat >= 0 else 32700 + zone_number
    return epsg_code

def generate_grid_utm(lonmin, latmin, lonmax, latmax, dx, dy):
    """
    des:
        Generate an equidistant spatial grid of a specified size based on UTM projection, and convert the result back to the WGS84 coordinate system.
    
    para:
        lonmin, latmin, lonmax, latmax : float   
            The latitude and longitude bounding box of the study area.
        dx, dy : float
            Grid width and height (in meters) in UTM projection.    
    return:
        geopandas.GeoDataFrame
            A GeoDataFrame dataset (EPSG:4326) containing the generated grid polygons.
    """
    tiles = []
    current_lat = latmin
    
    while current_lat < latmax:
        current_lon = lonmin
        row_max_lat = current_lat
        
        while current_lon < lonmax:
            epsg = get_utm_epsg(current_lon, current_lat)
            # Convert WGS84 to UTM
            utm_x, utm_y = coor2coor(4326, epsg, current_lon, current_lat)
            utm_x2 = utm_x + dx
            utm_y2 = utm_y + dy
            
            poly_utm = box(utm_x, utm_y, utm_x2, utm_y2)
            poly_utm_buffered = poly_utm.buffer(2000) # gap fill
            
            # Convert UTM boundary back to WGS84
            x_utm_coords, y_utm_coords = poly_utm_buffered.exterior.xy # gets boundary coordinates
            lon_coords, lat_coords = coor2coor(epsg, 4326, list(x_utm_coords), list(y_utm_coords))
            poly_wgs84 = Polygon(zip(lon_coords, lat_coords))
            
            tiles.append({'geometry': poly_wgs84, 'proj': f"EPSG:{epsg}"})
            
            # Calculate next grid start point
            next_lon, next_lat = coor2coor(epsg, 4326, utm_x2, utm_y2)
            current_lon = next_lon
            
            if next_lat > row_max_lat:
                row_max_lat = next_lat
                
        current_lat = row_max_lat
        
    return gpd.GeoDataFrame(tiles, crs="EPSG:4326")

In [4]:
def generate_grid_wgs84(xmin, ymin, xmax, ymax, dx=1, dy=1):
    """
    des:
        Generate a latitude and longitude grid (1° x 1°) based on WGS84.
    para:
        xmin, ymin, xmax, ymax: float
            The latitude and longitude boundaries of the study area, representing the minimum longitude, minimum latitude, 
            maximum longitude, and maximum latitude, respectively.
        dx: float, default value 1
            The width step of the grid in the longitude direction (unit: degrees, default 1°).
        dy: float, default value 1
            The height step of the grid in the latitude direction (unit: degrees, default 1°).
    return:
        geopandas.GeoDataFrame
    """
    tiles = []
    x_vals = np.arange(xmin, xmax, dx)  
    y_vals = np.arange(ymin, ymax, dy)
    
    for x in x_vals:
        for y in y_vals:
            tiles.append(box(x, y, x + dx, y + dy))
            
    return gpd.GeoDataFrame(geometry=tiles, crs="EPSG:4326")

print("Generating grids(WGS84)")
tb_tiles = generate_grid_utm(minx, miny, maxx, maxy, dx_utm, dy_utm)
tb_tiles_wgs84 = generate_grid_wgs84(minx, miny, maxx, maxy, 1, 1)


Generating grids(WGS84)


In [5]:
def filter_and_calc_area(gdf_tiles, study_area_gdf):
    """
    des:
        Removes empty grid cells (tiles outside the study area) and calculates the exact area of ​​each tile.
    Para:
        gdf_tiles : geopandas.GeoDataFrame
            The generated initial grid dataset.
        study_area_gdf : geopandas.GeoDataFrame
            The study area boundary dataset.
    return:
        geopandas.GeoDataFrame
            A cropped grid dataset with unique tile_ids and area information.
    """
    # Keep only intersecting tiles
    intersected = gpd.sjoin(gdf_tiles, study_area_gdf, how="inner", predicate="intersects") # spatial join
    intersected = intersected[gdf_tiles.columns].drop_duplicates()
    
    # Calculate area in an equal-area projection
    intersected['area_km2'] = intersected.geometry.to_crs("EPSG:6933").area / 1e6
    
    # add tile_id 
    intersected = intersected.reset_index(drop=True)
    intersected['tile_id'] = [str(i + 1).zfill(3) for i in range(len(intersected))]
    
    return intersected

print("Filtering intersections...")
tb_tiles = filter_and_calc_area(tb_tiles, tb_area)
tb_tiles_wgs84 = filter_and_calc_area(tb_tiles_wgs84, tb_area)


Filtering intersections...


In [6]:
def add_buffer_to_tiles(gdf_tiles, dist, is_degree=False):
    """
    des:
        Adds an outer buffer region to each tile. Supports both meters (for UTM) and degrees (for WGS84).
        Dynamically calculates the UTM EPSG code if the 'proj' column is missing.
    Para:
        gdf_tiles : geopandas.GeoDataFrame
            The filtered grid dataset.
        dist : float
            The distance to buffer outward.
        is_degree : bool, default False
            If True, treats `dist` as degrees (direct WGS84 buffer). 
            If False, treats `dist` as meters (projects to UTM first).
    return:
        geopandas.GeoDataFrame
            Dataset containing the expanded, rectangular buffered geometry.
    """
    buffered_geometries = []
    
    for idx, row in gdf_tiles.iterrows():
        geom = row.geometry
        
        if is_degree:
            # Buffer directly in degrees
            buffered_geom = geom.buffer(dist).envelope  # gets rectangular bounding box
            buffered_geometries.append(buffered_geom)
        else:
            # Buffer in meters via UTM projection
            if 'proj' in gdf_tiles.columns and pd.notna(row['proj']):
                epsg = row['proj']
            else:
                # Dynamic fallback: calculate UTM from polygon centroid
                centroid = geom.centroid  # gets center point
                epsg_int = get_utm_epsg(centroid.x, centroid.y)
                epsg = f"EPSG:{epsg_int}"
                
            single_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326")
            single_utm = single_gdf.to_crs(epsg)
            
            buffered_utm = single_utm.geometry.buffer(dist).envelope
            buffered_wgs84 = gpd.GeoDataFrame(geometry=buffered_utm, crs=epsg).to_crs("EPSG:4326")
            
            buffered_geometries.append(buffered_wgs84.geometry.iloc[0])
            
    # Assemble output dataset
    gdf_buf = gdf_tiles.copy()
    gdf_buf['geometry'] = buffered_geometries
    gdf_buf['area_km2'] = gdf_buf.geometry.to_crs("EPSG:6933").area / 1e6
    
    return gdf_buf

print("Applying buffers...")
# Buffer in meters for UTM grid
tb_tiles_buf = add_buffer_to_tiles(tb_tiles, buffer_m, is_degree=False)

# Buffer in degrees for WGS84 grid
tb_tiles_wgs84_buf = add_buffer_to_tiles(tb_tiles_wgs84, buffer_deg, is_degree=True)

Applying buffers...


In [7]:
out_dir = "data"
os.makedirs(out_dir, exist_ok=True)
# #Export UTM buffered tiles
path_buf = os.path.join(out_dir, 'hma_tiles_buf.gpkg')
tb_tiles_buf.to_file(path_buf, driver='GPKG', layer='hma_tiles_buf')
print(f"Exported: {os.path.abspath(path_buf)}")

#Export WGS84 buffered tiles
path_wgs84_buf = os.path.join(out_dir, 'hma_tiles_wgs84_buf.gpkg')
tb_tiles_wgs84_buf.to_file(path_wgs84_buf, driver='GPKG', layer='hma_tiles_wgs84_buf')
print(f"Exported WGS84 Grids: {os.path.abspath(path_wgs84_buf)}")

print("All tasks finished successfully!")

Exported: d:\A_WaterCode\HMA-GIS-Data\data\hma_tiles_buf.gpkg
Exported WGS84 Grids: d:\A_WaterCode\HMA-GIS-Data\data\hma_tiles_wgs84_buf.gpkg
All tasks finished successfully!
